In [0]:
# Shared functions used by all three silver notebooks -- loaded with %run ./00_silver_utils. Referential-integrity checking against parent tables is intentionally NOT included here.
import json
import logging
from datetime import datetime

from delta.tables import DeltaTable
from pyspark.sql import Row, DataFrame
from pyspark.sql.functions import (
    col, when, trim, upper, regexp_replace, length, lit, expr,
    current_timestamp, sha2, concat_ws, coalesce
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, TimestampType
)

In [0]:
# Sets up one shared logger reused by all three silver notebooks, same pattern as the bronze notebook.
logger = logging.getLogger("cdr_silver")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Silver utils loaded.")

In [0]:
# Fetches the run_id that bronze_ingest generated via Databricks Jobs task values, so bronze and every silver notebook log against the same run_id; falls back to a fresh id (with a warning) if run standalone.
import uuid

def get_shared_run_id(dbutils, upstream_task_key: str = "bronze_ingest") -> str:
    try:
        return dbutils.jobs.taskValues.get(taskKey=upstream_task_key, key="run_id")
    except Exception as e:
        fallback = str(uuid.uuid4())
        logger.warning(
            f"Could not read run_id from task '{upstream_task_key}' ({e}). "
            f"Generating a standalone run_id={fallback} -- this run will NOT share an "
            f"audit trail with bronze."
        )
        return fallback

In [0]:
# Reads a bronze table and drops bronze's own lineage/key columns, since silver adds its own and should never hash bronze's internal row-hash fallback key into a silver surrogate key.
BRONZE_LINEAGE_COLS = ["_ingest_timestamp", "_source_file", "_run_id", "_row_key"]

def read_bronze(spark, table_name: str, catalog: str = "cdr_bronze", schema: str = "raw_mimic_iv") -> DataFrame:
    df = spark.table(f"{catalog}.{schema}.{table_name}")
    return df.drop(*[c for c in BRONZE_LINEAGE_COLS if c in df.columns])

In [0]:
# Generic cleaning functions, each implementing one repeated pattern from the cleaning guide rather than a single column.
def fill_missing(df: DataFrame, column: str, value) -> DataFrame:
    """Replace null / empty-string values in a column with a fixed placeholder.
    Used for: admissions.admission_location/discharge_location/language/marital_status,
    emar_detail's Y/NaN flag columns, labevents.flag."""
    return df.withColumn(
        column,
        when(col(column).isNull() | (trim(col(column).cast("string")) == ""), lit(value))
        .otherwise(col(column))
    )


def standardize_category(df: DataFrame, column: str) -> DataFrame:
    """Trim whitespace, collapse internal double-spaces, drop a single trailing stray '.',
    and upper-case a category column so 'UNKNOWN', 'unknown ', ' Unknown' normalise to one value.
    Used for: transfers.careunit, pharmacy.dispensation, d_items.unitname,
    inputevents.amountuom/ordercomponenttypedescription, procedureevents.location."""
    cleaned = regexp_replace(trim(col(column)), r"\s+", " ")
    cleaned = regexp_replace(cleaned, r"\.$", "")
    return df.withColumn(column, upper(cleaned))


def rename_column(df: DataFrame, old_name: str, new_name: str) -> DataFrame:
    """Used for: d_icd_diagnoses/d_icd_procedures.long_title -> description,
    hcpcsevents.hcpcs_cd -> code, hcpcsevents.short_description -> description."""
    return df.withColumnRenamed(old_name, new_name)


def coalesce_and_drop(df: DataFrame, col_a: str, col_b: str, new_col: str) -> DataFrame:
    """Pick whichever of two text columns has more content, put it in new_col, drop both
    originals. Used for: d_hcpcs.long_description / short_description -> description."""
    df = df.withColumn(
        new_col,
        when(length(coalesce(col(col_a), lit(""))) >= length(coalesce(col(col_b), lit(""))), col(col_a))
        .otherwise(col(col_b))
    )
    return df.drop(col_a, col_b)


def drop_columns(df: DataFrame, columns: list) -> DataFrame:
    """Used for: labevents.storetime/value."""
    return df.drop(*[c for c in columns if c in df.columns])


def strip_html(df: DataFrame, column: str) -> DataFrame:
    """Remove HTML tags like <b>...</b> from a free-text column. Used for: poe.order_subtype."""
    return df.withColumn(column, regexp_replace(col(column), r"<[^>]+>", ""))


def cast_boolean(df: DataFrame, column: str) -> DataFrame:
    """Used for: datetimeevents.warning (currently an int flag column)."""
    return df.withColumn(column, col(column).cast("boolean"))


def map_values_to_default(df, column: str, values_to_map: list, default_value) -> DataFrame:
    """Replace any of values_to_map (case-insensitive) with default_value, leave everything else untouched.
    Used for: admissions.race (Unknown/Unable to Obtain/Patient declined to answer -> Unknown)."""
    return df.withColumn(
        column,
        when(upper(trim(col(column))).isin([v.upper() for v in values_to_map]), lit(default_value))
        .otherwise(col(column))
    )


def set_constant(df, column: str, value) -> DataFrame:
    """Unconditionally overwrite every value in a column with a fixed literal (not just nulls -- see fill_missing for that).
    Used for: datetimeevents.valueuom -> 'Date and Time'."""
    return df.withColumn(column, lit(value))

In [0]:
# Applies whatever rules silver_config.json lists for a table, in order, by name -- this is what lets a table's cleaning steps be edited in JSON instead of code.
FN_REGISTRY = {
    "fill_missing": fill_missing,
    "standardize_category": standardize_category,
    "rename_column": rename_column,
    "coalesce_and_drop": coalesce_and_drop,
    "drop_columns": drop_columns,
    "strip_html": strip_html,
    "cast_boolean": cast_boolean,
    "map_values_to_default": map_values_to_default,
    "set_constant": set_constant,
}


def apply_cleaning_rules(df: DataFrame, table_name: str, config: dict) -> DataFrame:
    rules = config.get(table_name, {}).get("cleaning_rules", [])
    for rule in rules:
        fn = FN_REGISTRY[rule["fn"]]
        df = fn(df, **rule["args"])
    return df

In [0]:
# For tables with no natural key (prescriptions, datetimeevents, outputevents, chartevents), hashes the row's business columns into a stable _natural_key so the same record gets the same key on every run.
def generate_surrogate_key(df: DataFrame, business_cols: list, key_name: str = "_natural_key") -> DataFrame:
    return df.withColumn(
        key_name,
        sha2(concat_ws("||", *[coalesce(col(c).cast("string"), lit("\u0000")) for c in business_cols]), 256)
    )

In [0]:
# Checks that the merge key is actually unique before a merge is attempted, same check bronze uses.
def check_key_uniqueness(df: DataFrame, key_cols: list):
    """Returns (is_unique, duplicate_row_count) for the given key columns on df."""
    total = df.count()
    distinct = df.dropDuplicates(key_cols).count()
    dup_count = total - distinct
    return dup_count == 0, dup_count

In [0]:
# Writes to silver: first run overwrites and creates the table, every run after that merges on the table's real key (never a per-run row hash, since silver always has a stable key by this point).
def write_silver(spark, df: DataFrame, target_table: str, merge_keys: list, run_id: str,
                  catalog: str = "cdr_silver", schema: str = "processed_mimic_iv"):
    full_target = f"{catalog}.{schema}.{target_table}"
    table_exists = spark.catalog.tableExists(full_target)

    df = df.withColumn("_silver_timestamp", current_timestamp()).withColumn("_run_id", lit(run_id))

    if not table_exists:
        is_unique, dup_count = check_key_uniqueness(df, merge_keys)
        if not is_unique:
            logger.warning(
                f"{full_target}: {dup_count} duplicate-key row(s) on {merge_keys} "
                f"-- deduplicating before initial write"
            )
            df = df.dropDuplicates(merge_keys)
        (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
           .saveAsTable(full_target))
        row_count = df.count()
        return {"load_type": "INITIAL_LOAD", "target_table": full_target,
                "row_count": row_count, "rows_inserted": row_count, "rows_updated": 0}

    is_unique, dup_count = check_key_uniqueness(df, merge_keys)
    if not is_unique:
        logger.warning(
            f"{full_target}: {dup_count} duplicate-key row(s) on {merge_keys} "
            f"-- deduplicating before merge"
        )
        df = df.dropDuplicates(merge_keys)
        is_unique, dup_count = check_key_uniqueness(df, merge_keys)
        if not is_unique:
            raise ValueError(
                f"Merge key {merge_keys} is still not unique for {full_target} after "
                f"deduplication ({dup_count} duplicate rows) -- investigate the source data."
            )

    delta_target = DeltaTable.forName(spark, full_target)
    merge_condition = " AND ".join([f"t.{c} <=> s.{c}" for c in merge_keys])

    (delta_target.alias("t")
        .merge(df.alias("s"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

    metrics = delta_target.history(1).select("operationMetrics").first()["operationMetrics"]
    row_count = df.count()
    return {
        "load_type": "INCREMENTAL_MERGE", "target_table": full_target, "row_count": row_count,
        "rows_inserted": int(metrics.get("numTargetRowsInserted", 0)),
        "rows_updated": int(metrics.get("numTargetRowsUpdated", 0)),
    }

In [0]:
# Defines one shared silver audit table (same shape as bronze's, plus a layer column) and creates it if it does not exist yet.
SILVER_AUDIT_TABLE = "cdr_silver.processed_mimic_iv.silver_audit_log"

SILVER_AUDIT_SCHEMA = StructType([
    StructField("run_id",           StringType(),    True),
    StructField("layer",            StringType(),    True),
    StructField("source_table",     StringType(),    True),
    StructField("target_table",     StringType(),    True),
    StructField("load_type",        StringType(),    True),
    StructField("status",           StringType(),    True),
    StructField("row_count",        LongType(),      True),
    StructField("rows_inserted",    LongType(),      True),
    StructField("rows_updated",     LongType(),      True),
    StructField("start_time",       TimestampType(), True),
    StructField("end_time",         TimestampType(), True),
    StructField("duration_seconds", DoubleType(),    True),
    StructField("error_message",    StringType(),    True),
])


def ensure_silver_audit_table(spark):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_AUDIT_TABLE} (
        run_id STRING, layer STRING, source_table STRING, target_table STRING,
        load_type STRING, status STRING, row_count BIGINT, rows_inserted BIGINT,
        rows_updated BIGINT, start_time TIMESTAMP, end_time TIMESTAMP,
        duration_seconds DOUBLE, error_message STRING
    ) USING DELTA
    """)


def write_audit_rows(spark, audit_rows: list):
    if not audit_rows:
        return
    audit_df = spark.createDataFrame(audit_rows, schema=SILVER_AUDIT_SCHEMA)
    audit_df.write.format("delta").mode("append").saveAsTable(SILVER_AUDIT_TABLE)
    logger.info(f"Silver audit log written: {len(audit_rows)} row(s) -> {SILVER_AUDIT_TABLE}")

In [0]:
# The one function every silver notebook calls per table: read bronze, clean, add a surrogate key if configured, write/merge to silver, and return an audit row.
def process_table(spark, table_name: str, config: dict, run_id: str, layer: str,
                   special_transform=None):
    start_time = datetime.now()
    table_cfg = config[table_name]
    try:
        df = read_bronze(spark, table_name)
        df = apply_cleaning_rules(df, table_name, config)

        if special_transform is not None:
            df = special_transform(df)

        if table_cfg["needs_surrogate_key"]:
            business_cols = [c for c in df.columns if c not in ("_silver_timestamp", "_run_id")]
            df = generate_surrogate_key(df, business_cols)

        result = write_silver(spark, df, table_cfg["target_table"], table_cfg["merge_keys"], run_id)
        end_time = datetime.now()

        logger.info(
            f"SUCCESS | {table_name} -> {result['target_table']} | {result['load_type']} | "
            f"{result['row_count']} rows | +{result['rows_inserted']} inserted / "
            f"~{result['rows_updated']} updated"
        )
        return Row(
            run_id=run_id, layer=layer, source_table=table_name, target_table=result["target_table"],
            load_type=result["load_type"], status="SUCCESS", row_count=result["row_count"],
            rows_inserted=result["rows_inserted"], rows_updated=result["rows_updated"],
            start_time=start_time, end_time=end_time,
            duration_seconds=(end_time - start_time).total_seconds(), error_message=None,
        )
    except Exception as e:
        end_time = datetime.now()
        error_message = str(e)[:2000]
        logger.error(f"FAILED | {table_name} | {error_message}")
        return Row(
            run_id=run_id, layer=layer, source_table=table_name, target_table=table_cfg.get("target_table"),
            load_type=None, status="FAILED", row_count=None, rows_inserted=None, rows_updated=None,
            start_time=start_time, end_time=end_time,
            duration_seconds=(end_time - start_time).total_seconds(), error_message=error_message,
        )